# Circular $R^2$ summary: Figure 5 panel E

Copied from `Figure_5_option/r2_distribution_summary.ipynb`; uses the local `slope_r2_helper.py` and writes into the Figure 5 SVG (`unifying_analysis_translationTrue.svg`).

The same datasets as `slope_vs_r2_summary.ipynb`, but this version plots only the circular $R^2$ to the mean model.
Each dataset is one column in the style of Figure 5 panel D (`fpl.scatter_box_violin_distribution`): jittered points,
a median line, and a shaded bootstrapped 95% CI of the median.

**$R^2$ window: each trajectory's own fit ROI** (`R2_WINDOW = 'fit_roi'`, option D in `surge_r2_window/`). The mean model is unchanged (circular mean of all aligned affine fits), but each trajectory is scored over the ~2 s its fit used rather than aligned −1 to 1 s, which at higher wind speeds can reach back before the fitted data into the post-flash surge. No period shifts are used. Set `R2_WINDOW = 'aligned'` to get the previous version. The $R^2$ axis is symmetric about 0: $-\ln(1-R^2)$ for $R^2 < 0$
(down to $R^2 \approx -244$), and $R^2 \ge 0$ stretched linearly to fill the same height above 0. A thin gray line marks $R^2 = 0$.
Settings: `R2_DIST_POS_SCALE`, `R2_DIST_ZERO_LINE` in `slope_r2_helper.py`.

Each panel is its own figurefirst figure (and layer) in `unifying_analysis_translationTrue.svg`:

1. **`r2_fig5_flash`**: flash, still air / unsteady / laminar (40 cm/s), in Figure 5 colors
2. **`r2_varwind_flash`**: flash, all 6 variable-wind speeds 
3. **`r2_sham_wt`**: sham (variable wind and still air/unsteady/laminar) plus WT flash and WT sham
4. **`r2_shuffled_synthetic`**: shuffled and synthetic controls

Run **Setup** first. After that, each section runs on its own (compute, load SVG, plot, write its own layer, save).

## Setup

In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']  # same as Figure 5 notebooks
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

from splitflow.set_zorder_functions import set_selective_rasterization

import slope_r2_helper as h

FIGURE_NAME = h.R2_FIGURE_NAME
TRANSLATION = h.TRANSLATION
R2_WINDOW = 'fit_roi'  # option D: score each trajectory over its own fit ROI ('aligned' = previous -1..1 s)


def add_group_labels(ax, groups, y=-0.45):
    """Text under the tick labels, centered on each group of columns. groups: list of (label, [x positions])."""
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    for label, xs in groups:
        ax.text(np.mean(xs), y, label, transform=trans, ha='center', va='top', fontsize=6)

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cpu


## Panel 1: flash, still air / unsteady / laminar (40 cm/s)

Order and colors match Figure 5 panel D.

In [2]:
FIFI_FIGURE = 'r2_fig5_flash'

# windtype -> (unifying results file, color, tick label)
PANEL1 = {'stillair': ('stillair_data_flash_translation%s.parquet' % TRANSLATION,        '#084a72', 'Still air'),
          'unsteady': ('unsteady_top_on_data_flash_translation%s.parquet' % TRANSLATION, '#2b75b3', 'Unsteady'),
          'laminar':  ('laminar_data_flash_translation%s.parquet' % TRANSLATION,        '#991128', 'Laminar\n(40 cm/s)')}

panel1_results = {}
for windtype, (fname, color, _) in PANEL1.items():
    unifying_filename = h.unifying_results_path(fname)
    panel1_results[windtype] = h.compute_slope_r2(unifying_filename, h.fly_trajec_filename(unifying_filename),
                                                  label='flash ' + windtype, r2_window=R2_WINDOW)

flash stillair               n =  200 (dropped 0)  median slope = 3.80  median R2 = 0.65  outside limits: 1


flash unsteady               n =   60 (dropped 0)  median slope = 4.22  median R2 = 0.63  outside limits: 0


flash laminar                n =  145 (dropped 0)  median slope = 3.96  median R2 = 0.47  outside limits: 0


In [3]:
layout = h.load_layout(FIGURE_NAME)
ax = layout.axes[(FIFI_FIGURE, 'ax')]
np.random.seed(h.RNG_SEED)  # jitter and bootstrap CI

xs = np.arange(1, len(PANEL1) + 1)
for x, (windtype, (_, color, _)) in zip(xs, PANEL1.items()):
    h.plot_r2_distribution(ax, x, panel1_results[windtype], color)

h.clean_r2_labels(ax, xs, [v[2] for v in PANEL1.values()], xlim=(0.5, len(PANEL1) + 0.5),
                  rotation=45, ticklabel_colors=[v[1] for v in PANEL1.values()])
set_selective_rasterization(ax, rasterize_markers=['o'])

h.save_figure(layout, FIFI_FIGURE, FIGURE_NAME)

## Panel 2: flash, variable wind (all speeds)

No period shifts: with the fit-ROI window they are not needed (the 60 cm/s `[0, 1]` exception in `E_slope_vs_rmse_var_wind` addresses the same surge problem).

In [4]:
FIFI_FIGURE = 'r2_varwind_flash'
PANEL2_COLORS = {speed: (h.VARWIND_HIGHLIGHT_COLOR if speed == h.VARWIND_HIGHLIGHT_SPEED else h.VARWIND_COLOR)
                 for speed in h.VARWIND_SPEEDS}  # light purple, 20 cm/s highlighted (as in panels G-I)

panel2_results = {}
for speed in h.VARWIND_SPEEDS:
    panel2_results[speed] = h.compute_slope_r2(h.varwind_unifying_filename('flash', speed), h.VARWIND_TRAJEC_FILENAME,
                                               label='var wind flash %d cm/s' % h.VARWIND_SPEED_LABELS[speed],
                                               r2_window=R2_WINDOW)

var wind flash 0 cm/s        n =  700 (dropped 0)  median slope = 3.79  median R2 = 0.85  outside limits: 3


var wind flash 3 cm/s        n =  491 (dropped 0)  median slope = 3.62  median R2 = 0.74  outside limits: 1


var wind flash 10 cm/s       n =  257 (dropped 0)  median slope = 3.60  median R2 = 0.62  outside limits: 0


var wind flash 20 cm/s       n =  700 (dropped 0)  median slope = 3.54  median R2 = 0.63  outside limits: 0


var wind flash 30 cm/s       n =  272 (dropped 0)  median slope = 3.47  median R2 = 0.46  outside limits: 2


var wind flash 60 cm/s       n =  241 (dropped 0)  median slope = 3.60  median R2 = 0.35  outside limits: 0


In [5]:
layout = h.load_layout(FIGURE_NAME)
ax = layout.axes[(FIFI_FIGURE, 'ax')]
np.random.seed(h.RNG_SEED)

xs = np.arange(1, len(h.VARWIND_SPEEDS) + 1)
for x, speed in zip(xs, h.VARWIND_SPEEDS):
    h.plot_r2_distribution(ax, x, panel2_results[speed], PANEL2_COLORS[speed])

h.clean_r2_labels(ax, xs, [str(h.VARWIND_SPEED_LABELS[s]) for s in h.VARWIND_SPEEDS], xlim=(0.5, len(xs) + 0.5),
                  show_ylabel=False, xlabel='Wind speed (cm/s)')
set_selective_rasterization(ax, rasterize_markers=['o'])

h.save_figure(layout, FIFI_FIGURE, FIGURE_NAME)

## Panel 3: sham and WT controls

One column per dataset, in four groups separated by a small gap:
- sham, variable wind, 6 speeds
- sham, still air / unsteady / laminar (40 cm/s)
- WT flash, still air / unsteady / laminar (unsteady WT uses the `experiencedunsteady_` files, as in `sup_fig_unifying_analysis`)
- WT sham, still air / unsteady / laminar

In [6]:
FIFI_FIGURE = 'r2_sham_wt'
GROUP_GAP = 0.75  # extra space between groups, in column units
FLY_TICKLABELS = {'stillair': 'Still air', 'unsteady': 'Unsteady', 'laminar': 'Laminar'}

# group label -> {windtype: (unifying results file, color)}; colors per dataset so they can be changed later
PANEL3_FLIES = {
    'Sham': {'stillair': ('stillair_data_sham_translation%s.parquet' % TRANSLATION,                    'gray'),
             'unsteady': ('unsteady_top_on_data_sham_AUGMENTED_translation%s.parquet' % TRANSLATION,   'gray'),
             'laminar':  ('laminar_data_sham_translation%s.parquet' % TRANSLATION,                     'gray')},
    'WT flash': {'stillair': ('WT_stillair_data_flash_translation%s.parquet' % TRANSLATION,                    'gray'),
                 'unsteady': ('experiencedunsteady_top_on_data_WT_flash_translation%s.parquet' % TRANSLATION,  'gray'),
                 'laminar':  ('WT_laminar_data_flash_translation%s.parquet' % TRANSLATION,                     'gray')},
    'WT sham': {'stillair': ('WT_stillair_data_sham_translation%s.parquet' % TRANSLATION,                    'gray'),
                'unsteady': ('experiencedunsteady_top_on_data_WT_sham_translation%s.parquet' % TRANSLATION,  'gray'),
                'laminar':  ('WT_laminar_data_sham_translation%s.parquet' % TRANSLATION,                     'gray')},
}
PANEL3_VARWIND_SHAM_COLORS = {speed: 'gray' for speed in h.VARWIND_SPEEDS}

# group label -> list of (tick label, results, color), in plotting order
VARWIND_SHAM_GROUP = 'Var. wind sham (cm/s)'
panel3_results = {VARWIND_SHAM_GROUP: []}
for speed in h.VARWIND_SPEEDS:
    r = h.compute_slope_r2(h.varwind_unifying_filename('sham', speed), h.VARWIND_TRAJEC_FILENAME,
                           label='var wind sham %d cm/s' % h.VARWIND_SPEED_LABELS[speed],
                           r2_window=R2_WINDOW)
    panel3_results[VARWIND_SHAM_GROUP].append((str(h.VARWIND_SPEED_LABELS[speed]), r, PANEL3_VARWIND_SHAM_COLORS[speed]))
for group, datasets in PANEL3_FLIES.items():
    panel3_results[group] = []
    for windtype, (fname, color) in datasets.items():
        unifying_filename = h.unifying_results_path(fname)
        r = h.compute_slope_r2(unifying_filename, h.fly_trajec_filename(unifying_filename), label=group + ' ' + windtype,
                               r2_window=R2_WINDOW)
        panel3_results[group].append((FLY_TICKLABELS[windtype], r, color))

var wind sham 0 cm/s         n =  388 (dropped 0)  median slope = 1.77  median R2 = -0.79  outside limits: 1


var wind sham 3 cm/s         n =  423 (dropped 0)  median slope = 1.87  median R2 = -1.24  outside limits: 2


var wind sham 10 cm/s        n =  241 (dropped 0)  median slope = 1.89  median R2 = -1.42  outside limits: 1


var wind sham 20 cm/s        n =  225 (dropped 0)  median slope = 1.92  median R2 = -1.48  outside limits: 1


var wind sham 30 cm/s        n =  232 (dropped 0)  median slope = 1.95  median R2 = -0.55  outside limits: 1
var wind sham 60 cm/s        n =   68 (dropped 0)  median slope = 2.17  median R2 = 0.01  outside limits: 0


Sham stillair                n =  192 (dropped 0)  median slope = 1.80  median R2 = -0.57  outside limits: 0
Sham unsteady                n =   85 (dropped 0)  median slope = 4.88  median R2 = -0.29  outside limits: 16


Sham laminar                 n =  117 (dropped 0)  median slope = 2.59  median R2 = -0.14  outside limits: 1
WT flash stillair            n =   54 (dropped 0)  median slope = 1.10  median R2 = -2.65  outside limits: 2
WT flash unsteady            n =   68 (dropped 0)  median slope = 3.52  median R2 = -0.08  outside limits: 3


WT flash laminar             n =  138 (dropped 0)  median slope = 1.34  median R2 = -0.51  outside limits: 2
WT sham stillair             n =   68 (dropped 0)  median slope = 1.07  median R2 = -1.85  outside limits: 3
WT sham unsteady             n =   66 (dropped 0)  median slope = 3.63  median R2 = 0.04  outside limits: 2


WT sham laminar              n =  127 (dropped 0)  median slope = 1.19  median R2 = -1.24  outside limits: 0


In [7]:
layout = h.load_layout(FIGURE_NAME)
ax = layout.axes[(FIFI_FIGURE, 'ax')]
np.random.seed(h.RNG_SEED)

VARWIND_UNITS_LABEL_Y = -0.3  # axes fraction, below the rotated speed tick labels
xticks, xticklabels, group_positions = [], [], []
x = 1.0
for group, columns in panel3_results.items():
    xs_group = []
    for ticklabel, r, color in columns:
        h.plot_r2_distribution(ax, x, r, color)
        xticks.append(x); xticklabels.append(ticklabel); xs_group.append(x)
        x += 1
    group_positions.append((group, xs_group))
    x += GROUP_GAP

h.clean_r2_labels(ax, xticks, xticklabels, xlim=(0.5, xticks[-1] + 0.5), show_ylabel=False, rotation=45)
# Group names (Sham / WT flash / WT sham) are in the SVG above the panel; only the variable-wind
# speeds need a units label, placed under their (rotated) tick labels.
varwind_xs = dict(group_positions)[VARWIND_SHAM_GROUP]
add_group_labels(ax, [('Wind speed\n(cm/s)', varwind_xs)], y=VARWIND_UNITS_LABEL_Y)
set_selective_rasterization(ax, rasterize_markers=['o'])

h.save_figure(layout, FIFI_FIGURE, FIGURE_NAME)

## Panel 4: shuffled and synthetic controls

The six scenarios from panel C of `supplemental_unifying_analysis_shuffled_synthetic.svg`. Lévy flight has a median $R^2$ of about −68, so some of its points fall below the bottom of the axis.

In [8]:
FIFI_FIGURE = 'r2_shuffled_synthetic'

scenarios = h.shuffled_synthetic_scenarios()
PANEL4_COLORS = {key: 'gray' for key in scenarios.SCENARIOS}

panel4_results = {}
for key in scenarios.SCENARIOS:
    unifying_filename, trajec_filename = h.shuffled_synthetic_filenames(key)
    panel4_results[key] = h.compute_slope_r2(unifying_filename, trajec_filename, label=key, r2_window=R2_WINDOW)

shuffled_laminar             n =  200 (dropped 0)  median slope = 1.03  median R2 = -5.04  outside limits: 0


shuffled_stillair            n =  200 (dropped 0)  median slope = 1.34  median R2 = -3.53  outside limits: 3


levy_flight                  n =  200 (dropped 0)  median slope = 0.03  median R2 = -70.39  outside limits: 14


random_turns                 n =  200 (dropped 0)  median slope = 5.02  median R2 = 0.11  outside limits: 10


brownian                     n =  200 (dropped 0)  median slope = 10.54  median R2 = -0.11  outside limits: 152


run_and_tumble               n =  200 (dropped 0)  median slope = 9.66  median R2 = -0.09  outside limits: 137


In [9]:
layout = h.load_layout(FIGURE_NAME)
ax = layout.axes[(FIFI_FIGURE, 'ax')]
np.random.seed(h.RNG_SEED)

xs = np.arange(1, len(scenarios.SCENARIOS) + 1)
for x, key in zip(xs, scenarios.SCENARIOS):
    h.plot_r2_distribution(ax, x, panel4_results[key], PANEL4_COLORS[key])

h.clean_r2_labels(ax, xs, [scenarios.SHORT_LABELS[k] for k in scenarios.SCENARIOS], xlim=(0.5, len(xs) + 0.5),
                  show_ylabel=False, rotation=45)
set_selective_rasterization(ax, rasterize_markers=['o'])

h.save_figure(layout, FIFI_FIGURE, FIGURE_NAME)